# 06 · Going Live Without Docker

**Module 07 · Production & Observability**

The desk is reliable and observable. What stands between it and real customers?

| Concern | Section |
|---|---|
| Settings and secrets | 1 |
| Who may call it? | 2 |
| One caller using all the capacity | 3 |
| Where the service keeps its state | 4 |
| Starting it | 5 and 6 |
| Putting it on a server | 7 |
| Releasing changes safely | 8 and 9 |

**About Docker.** Docker packages an app and everything it needs into one image that runs the same everywhere. It is very common, but **you do not need it** to run a Python service. A hosting platform can build your project straight from GitHub using your `requirements.txt` and a start command, and that is the route we take.

In [1]:
import json
import os
import subprocess
import sys
import time

import httpx
from fastapi.testclient import TestClient

import desk_service
from reliability import RateLimiter

client = TestClient(desk_service.app)
print("Ready")

Ready


## 1. Settings and secrets
**Rule 1: nothing that changes between your laptop and the server goes in the code.** Keys, limits and switches come from **environment variables**.

| Variable | What it does | Default |
|---|---|---|
| `OPENAI_API_KEY` | Your OpenAI key. A **secret** | (required) |
| `DESK_ENV` | `production` switches on the safety checks below | `development` |
| `DESK_API_KEYS` | The keys customers' apps may use (comma separated). A **secret** | a public dev key |
| `DESK_ADMIN_KEYS` | The keys your staff may use. A **secret** | a public dev key |
| `DESK_RATE_LIMIT` | Requests per minute per caller | 30 |
| `DESK_MAX_COST` | Dollar budget per request | 0.02 |
| `DESK_JUDGE_SAMPLE_RATE` | Share of replies graded by the judge | 0.25 |
| `DESK_KILL_SWITCH` | `on` starts the service with the AI switched off | off |
| `DESK_LOG_DIR` | Where the logs go | `logs/` |
| `DESK_STORE_ON_OPENAI` | `yes` also stores calls in OpenAI's dashboard | no |

**Secrets** (keys) never go in git, in notebooks, in logs or in chat messages. Locally they live in `.env`, which `.gitignore` excludes. On a server you set them in the host's settings page.

**Rule 2: fail loudly at startup.** A server that starts with a *missing or well-known* secret is a security hole that nobody notices. Our service refuses:

*From `desk_service.py`:*

```python
ENVIRONMENT = os.getenv("DESK_ENV", "development")  # set DESK_ENV=production on a real server
RATE_LIMIT_PER_MINUTE = int(os.getenv("DESK_RATE_LIMIT", "30"))

# Two kinds of keys: client keys (customers' apps) and admin keys (your staff).
# The development keys below are public knowledge, because they are in this file. NEVER use them on a real server.
DEV_CLIENT_KEY = "dev-client-key"
DEV_ADMIN_KEY = "dev-admin-key"


def read_keys(env_name, development_key):
    """Read a comma separated list of keys from an environment variable."""
    keys = []
    for part in os.getenv(env_name, "").split(","):
        if part.strip() != "":
            keys.append(part.strip())

    if len(keys) == 0:
        keys.append(development_key)  # so the service works on your laptop with no setup
    return keys


CLIENT_KEYS = read_keys("DESK_API_KEYS", DEV_CLIENT_KEY)
ADMIN_KEYS = read_keys("DESK_ADMIN_KEYS", DEV_ADMIN_KEY)


def check_config():
    """Refuse to start a production server that still uses the public development keys."""
    if ENVIRONMENT == "production":
        if DEV_CLIENT_KEY in CLIENT_KEYS or DEV_ADMIN_KEY in ADMIN_KEYS:
            raise SystemExit("Set DESK_API_KEYS and DESK_ADMIN_KEYS before running with DESK_ENV=production.")


check_config()
```

In [2]:
# Start the service as if it were production, but without setting real keys. The service should refuse.
environment = dict(os.environ)
environment["DESK_ENV"] = "production"
environment.pop("DESK_API_KEYS", None)
environment.pop("DESK_ADMIN_KEYS", None)

result = subprocess.run([sys.executable, "-c", "import desk_service"], env=environment, capture_output=True, text=True)
print("Exit code:", result.returncode)
print("Message  :", result.stderr.strip().splitlines()[-1])

Exit code: 1
Message  : Set DESK_API_KEYS and DESK_ADMIN_KEYS before running with DESK_ENV=production.


## 2. Who may call it?
Without authentication, anyone who finds the address can spend your OpenAI money. We use **API keys** in the `X-API-Key` header, with two levels:

| Key | May do |
|---|---|
| A **client** key | Send emails, send feedback |
| An **admin** key | Also: approve refunds, read the metrics, flip the kill switch |

*From `desk_service.py`:*

```python
def require_client(x_api_key: str | None = Header(default=None)):
    """Customers' apps must send their key in the X-API-Key header. Returns a short name for the caller."""
    if x_api_key is None or x_api_key not in CLIENT_KEYS:
        raise HTTPException(status_code=401, detail="Missing or invalid API key.")
    return x_api_key[:6] + "..."  # never put a whole key in a log


def require_admin(x_api_key: str | None = Header(default=None)):
    """Admin endpoints need an admin key. A customer key is not enough (403 = known, but not allowed)."""
    if x_api_key is None:
        raise HTTPException(status_code=401, detail="Missing API key.")
    if x_api_key not in ADMIN_KEYS:
        raise HTTPException(status_code=403, detail="This endpoint needs an admin key.")
    return x_api_key[:6] + "..."
```

In [3]:
email = {"email": "Hello"}

for label, headers in [("no key", {}), ("a wrong key", {"X-API-Key": "nope"})]:
    print(label.ljust(24), client.post("/emails", json=email, headers=headers).status_code)

print("a client key on /approvals".ljust(24), client.get("/approvals", headers={"X-API-Key": "dev-client-key"}).status_code, "(403: known, but not allowed)")
print("an admin key on /approvals".ljust(24), client.get("/approvals", headers={"X-API-Key": "dev-admin-key"}).status_code)

no key                   401
a wrong key              401
a client key on /approvals 403 (403: known, but not allowed)
an admin key on /approvals 200


API keys are the simplest thing that works. For apps with users, you would add a login system (OAuth). That is a topic of its own. And always serve over **HTTPS**: hosting platforms do this for you.

## 3. Rate limiting
Even a *legitimate* caller can have a bug and send ten thousand emails a minute. The rate limiter allows each key a number of requests per minute, then answers `429 Too Many Requests`:

*From `reliability.py`:*

```python
class RateLimiter:
    """Allow each caller only 'limit' requests per 'seconds'. It protects the shop from one noisy client
    (a bug, an attack) using up the whole OpenAI budget."""

    def __init__(self, limit, seconds=60):
        self.limit = limit
        self.seconds = seconds
        self.calls = {}  # caller -> the times of their recent requests
        self.lock = threading.Lock()

    def allow(self, caller):
        """True if this caller may make a request now. False means: answer with HTTP 429 (too many requests)."""
        with self.lock:
            now = time.time()

            # Keep only the requests from the last 'seconds'
            recent = []
            for moment in self.calls.get(caller, []):
                if now - moment < self.seconds:
                    recent.append(moment)

            if len(recent) >= self.limit:
                self.calls[caller] = recent
                return False

            recent.append(now)
            self.calls[caller] = recent
            return True
```

In [4]:
real_limiter = desk_service.limiter
desk_service.limiter = RateLimiter(limit=3)     # a tiny limit, just for the demonstration

too_long = {"email": "x" * 2500}                # blocked by the length guard, so it costs nothing
for number in range(1, 6):
    response = client.post("/emails", json=too_long, headers={"X-API-Key": "dev-client-key"})
    print("Request", number, "-> HTTP", response.status_code)

desk_service.limiter = real_limiter             # back to normal

{"time": "2026-11-22T19:17:06+00:00", "level": "INFO", "event": "request_finished", "request_id": "af293a0ae719", "status": "ok", "duration_ms": 0, "model_calls": 0, "cost_usd": 0.0}


Request 1 -> HTTP 200
{"time": "2026-11-22T19:17:06+00:00", "level": "INFO", "event": "request_finished", "request_id": "797fc105d867", "status": "ok", "duration_ms": 0, "model_calls": 0, "cost_usd": 0.0}


Request 2 -> HTTP 200


{"time": "2026-11-22T19:17:06+00:00", "level": "INFO", "event": "request_finished", "request_id": "2e40e5f78f22", "status": "ok", "duration_ms": 0, "model_calls": 0, "cost_usd": 0.0}


Request 3 -> HTTP 200
Request 4 -> HTTP 429
Request 5 -> HTTP 429


The first three go through, the rest get `429`, with a `Retry-After` header that tells a well-behaved client when to try again. This protects your **wallet** as much as your server.

## 4. Where does the service keep its state? (An honest limit)
Look at what lives in memory in `desk_service.py`:

| State | Lost when the service restarts? | A second copy of the service sees it? |
|---|---|---|
| The idempotency answers | Yes | No |
| The rate limiter's counters | Yes | No |
| The refund list and the approval queue | **Yes** | No |
| The logs | No (they are files) | Only on the same machine |

**For learning and a small single-server shop this is fine. For a real business it is not**: a restart would forget which refunds are waiting for approval. The fix is a **database** (for the refunds and approvals) and a shared store such as **Redis** (for idempotency and rate limits). That is also what lets you run **several copies** of the service. Today the service must run as exactly **one process**, so do not turn on multiple workers.

Knowing the limits of what you built is part of shipping it.

## 5. Starting the service
```powershell
# On your laptop
python "Production and Observability/desk_service.py"

# The same thing, the way hosting platforms start Python web services
uvicorn desk_service:app --host 0.0.0.0 --port 8000
```
`uvicorn` is the program that listens for requests and hands them to your FastAPI app. `0.0.0.0` means "accept connections from outside this machine", which a server needs. The `--reload` option restarts on every code change (development only).

## 6. Running it for real
So far we used a pretend client. Let us start a **real server**, call it over a real network connection, and stop it. (It uses port 8765 on your own machine.)

In [5]:
server_environment = dict(os.environ)
server_environment["DESK_JUDGE_SAMPLE_RATE"] = "0"       # no judge calls for this short test

server = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "desk_service:app", "--port", "8765", "--log-level", "warning"],
    env=server_environment, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
)

# Wait until the server answers its health check
base = "http://127.0.0.1:8765"
for attempt in range(30):
    try:
        if httpx.get(base + "/health", timeout=2).status_code == 200:
            break
    except httpx.TransportError:
        time.sleep(0.5)

print("Health check:", httpx.get(base + "/health").json())

Health check: {'status': 'ok', 'prompt_version': '2026-11-22', 'code_version': '09ad2de'}


In [6]:
response = httpx.post(base + "/emails", json={"email": "How much does express shipping cost?"},
                      headers={"X-API-Key": "dev-client-key"}, timeout=60)
print("HTTP", response.status_code)
print(json.dumps(response.json(), indent=2))

HTTP 200
{
  "request_id": "3eb5f5149ad0",
  "status": "replied",
  "reply": "Express shipping always costs $12. This applies regardless of the order total.",
  "route": "policy",
  "note": "",
  "refunds_issued": [],
  "sent_for_approval": [],
  "cost_usd": 0.001771,
  "latency_ms": 5528
}


In [7]:
# The kill switch, on the real server: every email now goes to a person, and no model is called.
admin = {"X-API-Key": "dev-admin-key"}
print(httpx.post(base + "/kill-switch", json={"on": True}, headers=admin).json())

response = httpx.post(base + "/emails", json={"email": "Where is my order A1002?"}, headers={"X-API-Key": "dev-client-key"}, timeout=60)
print(response.json()["status"], "-", response.json()["note"], "- cost: $" + str(response.json()["cost_usd"]))

print(httpx.post(base + "/kill-switch", json={"on": False}, headers=admin).json())

{'kill_switch': True}


human - The desk is switched off, so a person will answer this email. - cost: $0.0


{'kill_switch': False}


In [8]:
server.terminate()   # stop the server
server.wait()
print("Server stopped")

Server stopped


That was a real web server answering real HTTP requests. The **kill switch** worked instantly, with no restart: the answer came back with no model calls and no cost.

## 7. Putting it on a server, without Docker
A **hosting platform** runs your code on a machine that is always on, with a public HTTPS address. Many of them (for example Render and Railway) can build a Python service **straight from your GitHub repository**: you do not install anything on the server yourself. Free tiers, prices and menus change, so check their current docs. The recipe is the same everywhere:

| Setting | What you enter |
|---|---|
| **Repository** | Your GitHub repo |
| **Root directory** | `Production and Observability` (the folder with `desk_service.py`) |
| **Build command** | `pip install -r requirements.txt` |
| **Start command** | `uvicorn desk_service:app --host 0.0.0.0 --port $PORT` (the host tells you the port in `$PORT`) |
| **Health check path** | `/health` |
| **Environment variables** | `OPENAI_API_KEY`, `DESK_ENV=production`, `DESK_API_KEYS`, `DESK_ADMIN_KEYS`, and the other settings from section 1 |
| **Instances / workers** | **One** (see section 4) |

I cannot create an account or deploy for you, and deploying is publishing: it puts a service that spends your OpenAI money on the internet. So this part is yours to do, when you are ready. Before you do, the checklist:

- [ ] `DESK_ENV=production`, with long random keys in `DESK_API_KEYS` and `DESK_ADMIN_KEYS`
- [ ] A **spending limit** set in your OpenAI account, as a second line of defence behind the per-request budget
- [ ] The rate limit and per-request budget are set to numbers you chose
- [ ] You know how to flip the **kill switch**, and who is on call
- [ ] The logs go somewhere you can read them (the platform's log viewer shows what the service prints)
- [ ] You ran the evals, and they pass (next section)
- [ ] You accept the limits of section 4 (one process, state in memory), or you moved the state to a database

## 8. A release gate: run the evals before every release
Module 05 gave you a test suite. The most valuable use of it is **automatic**: every change to the code runs the evals, and a failing run blocks the release.

GitHub can do this for you, for free, with a small file called a *workflow* (GitHub Actions). We saved an example as `ci/evals.yml.example`. It is deliberately **not active**, because it needs your OpenAI key stored as a GitHub secret and costs a few cents per run:

In [9]:
print(open("ci/evals.yml.example", encoding="utf-8").read())

# An EXAMPLE of a release gate: run the Module 05 evals automatically, and block the change if they fail.
#
# It is saved as .example on purpose, so it does NOT run on your repository by itself.
# To switch it on:
#   1. copy it to .github/workflows/evals.yml at the ROOT of your repository
#   2. in GitHub: Settings > Secrets and variables > Actions > New repository secret, name it OPENAI_API_KEY
#   3. check that the versions of the actions below are still current
#
# Every run makes about 100 small model calls (a few cents).

name: Desk evals

on:
  pull_request:
    paths:
      - "Production and Observability/**"
  workflow_dispatch:   # also lets you start it by hand from the Actions tab

jobs:
  evals:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4

      - uses: actions/setup-python@v5
        with:
          python-version: "3.13"

      - name: Install the dependencies
        run: pip install -r "Production and Observability/requirements.txt"

      -

`run_evals.py` exits with an error code when the pass rate is below its threshold, and that error code is what stops the release.

## 9. Releasing changes safely
| Practice | What it means |
|---|---|
| **Small releases** | A small change is easy to understand and to undo |
| **Evals first** | Never release a change that makes the pass rate fall (section 8) |
| **Versions in the logs** | You can tell which release caused a change in the numbers (notebook 3) |
| **Watch after releasing** | Look at the dashboard and alerts for the first hour |
| **A way back** | If the numbers get worse: flip the kill switch, then roll back with `git revert` and redeploy |
| **Gradual rollout** (when you grow) | Send 5% of traffic to the new version first, and compare |

## What this module did **not** cover
Being honest about this is part of production work:
- A **database** for the state in section 4, and running several copies for more capacity
- **Logins and OAuth**, if end users (not just apps) call the service
- **Hosted monitoring** tools (see notebook 3), and paging people
- **Docker and Kubernetes**, when you need identical environments or large scale
- **Load testing**: how many emails per minute can it really take?

## You made it
Look back at the whole course. You started with one API call. You built an agent loop, tools, memory, multiple agents, guardrails and evals. You compared frameworks. And now you can put the desk behind an API and answer, for any request, *what happened, how long it took, what it cost, and whether it was good*.

**Next: the capstone.** We will build the support desk in each framework from Module 06, put the Module 05 guardrails and this module's tracing around every one of them, score them all with the same evals, and compare real numbers.